# Generate Synthetic Clinical Data

This notebook generates:

1. **100 PDF files** – clinical notes for 35 patients → saved in volume subfolder **clinical_notes**
2. **1 JSON file** – lung cancer medical images metadata for the same 35 patients → volume root
3. **1 CSV file** – Electronic Health Records (EHR) for the same 35 patients → volume root

All outputs are written to the Unity Catalog volume **melissap.melissa_pang.project_volume** (from setup_volume_and_genie.ipynb). They share a common **patient_id** so records can be linked across files.

**Dependencies:** `faker`, `fpdf2`, `pandas`, `databricks-sdk`

In [9]:
# Run this cell first if you get ModuleNotFoundError for faker, fpdf2, pandas, or databricks-sdk
%pip install faker fpdf2 pandas databricks-sdk --quiet


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
# Configuration
NUM_PATIENTS = 35
NUM_PDF_NOTES = 100
SEED = 42
import os

# Volume from setup_volume_and_genie.ipynb: melissap.melissa_pang.project_volume
VOLUME_BASE = "/Volumes/melissap/melissa_pang/project_volume"
PDF_SUBDIR = f"{VOLUME_BASE}/clinical_notes"   # PDFs go in this subfolder
OUTPUT_DIR = VOLUME_BASE                         # JSON and CSV go in volume root (not in subfolder)

# Create clinical_notes subfolder in the volume (Databricks SDK)
from databricks.sdk import WorkspaceClient
w = WorkspaceClient(profile="DEFAULT")
w.files.create_directory(directory_path=PDF_SUBDIR)
print(f"Volume: {VOLUME_BASE}")
print(f"Clinical notes subfolder: {PDF_SUBDIR}")
print(f"JSON/CSV output: {OUTPUT_DIR}")

Volume: /Volumes/melissap/melissa_pang/project_volume
Clinical notes subfolder: /Volumes/melissap/melissa_pang/project_volume/clinical_notes
JSON/CSV output: /Volumes/melissap/melissa_pang/project_volume


In [11]:
# Generate unique patient IDs (used to link all outputs)
patient_ids = [f"PAT_{i:03d}" for i in range(1, NUM_PATIENTS + 1)]
print(f"Generated {len(patient_ids)} patient IDs: {patient_ids[:5]} ... {patient_ids[-2:]}")

Generated 35 patient IDs: ['PAT_001', 'PAT_002', 'PAT_003', 'PAT_004', 'PAT_005'] ... ['PAT_034', 'PAT_035']


In [12]:
# Assign each of the 100 clinical notes to a patient (some patients get multiple notes)
import random
random.seed(SEED)
note_assignments = random.choices(patient_ids, k=NUM_PDF_NOTES)
print(f"Assigned {NUM_PDF_NOTES} notes across patients. Sample: {note_assignments[:10]}")

Assigned 100 notes across patients. Sample: ['PAT_023', 'PAT_001', 'PAT_010', 'PAT_008', 'PAT_026', 'PAT_024', 'PAT_032', 'PAT_004', 'PAT_015', 'PAT_002']


In [14]:
# Generate 100 PDF clinical notes
import io
from faker import Faker
from fpdf import FPDF
from datetime import datetime, timedelta

Faker.seed(SEED)
fake = Faker()

def make_clinical_note(patient_id: str, note_index: int) -> str:
    """Build synthetic clinical note text."""
    visit_date = fake.date_between(start_date="-2y", end_date="today")
    chief = fake.random_element([
        "Cough and shortness of breath", "Chest pain", "Follow-up lung nodule",
        "Fatigue and weight loss", "Routine oncology follow-up", "Dyspnea on exertion"
    ])
    assessment = fake.random_element([
        "Stable lung malignancy. Continue current regimen.",
        "New nodule noted; recommend short-term CT follow-up.",
        "No interval change. Surveillance imaging as planned.",
        "Treatment-related fatigue; supportive care."
    ])
    return (
        f"CLINICAL NOTE\n"
        f"Patient ID: {patient_id}\n"
        f"Date: {visit_date}\n"
        f"Encounter: {note_index}\n\n"
        f"Chief Complaint: {chief}\n\n"
        f"Assessment: {assessment}\n\n"
        f"Plan: {fake.sentence()}\n"
        f"Next visit: {fake.date_between(start_date='today', end_date='+3m')}"
    )

class ClinicalNotePDF(FPDF):
    def header(self):
        self.set_font("Helvetica", "", 10)
        self.cell(0, 8, "Confidential - Synthetic Clinical Note", ln=True)
        self.ln(2)

    def footer(self):
        self.set_y(-12)
        self.set_font("Helvetica", "", 8)
        self.cell(0, 10, f"Generated synthetic data - Page {self.page_no()}", align="C")

for i, patient_id in enumerate(note_assignments):
    note_num = i + 1
    text = make_clinical_note(patient_id, note_num)
    pdf = ClinicalNotePDF()
    pdf.add_page()
    pdf.set_font("Helvetica", size=11)
    pdf.multi_cell(0, 7, text)
    pdf_bytes = pdf.output(dest="S")
    if isinstance(pdf_bytes, str):
        pdf_bytes = pdf_bytes.encode("latin-1")
    volume_path = f"{PDF_SUBDIR}/note_{note_num:03d}_patient_{patient_id}.pdf"
    w.files.upload(file_path=volume_path, contents=io.BytesIO(bytes(pdf_bytes)), overwrite=True)

print(f"Created {NUM_PDF_NOTES} PDF clinical notes in {PDF_SUBDIR}")

/var/folders/t7/0mw1_y2j5d96gbt8610s6pg80000gp/T/ipykernel_45376/2548000827.py:37: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  self.cell(0, 8, "Confidential - Synthetic Clinical Note", ln=True)
/var/folders/t7/0mw1_y2j5d96gbt8610s6pg80000gp/T/ipykernel_45376/2548000827.py:52: DeprecationWarning: "dest" parameter is deprecated since v2.2.0 and will be removed in a future release
  pdf_bytes = pdf.output(dest="S")


Created 100 PDF clinical notes in /Volumes/melissap/melissa_pang/project_volume/clinical_notes


In [15]:
# Generate lung cancer medical images metadata (1 JSON file for all 35 patients)
import io
import json

random.seed(SEED + 1)
image_modalities = ["CT", "CT", "CT", "MRI", "PET-CT"]  # CT weighted
findings = [
    "Primary lung mass LUL", "Nodule RLL", "Mediastinal lymphadenopathy",
    "Pleural effusion", "No interval change", "Ground glass opacity"
]

images_metadata = []
image_id = 1
for patient_id in patient_ids:
    n_images = random.randint(2, 6)
    for _ in range(n_images):
        study_date = fake.date_between(start_date="-2y", end_date="today").isoformat()
        images_metadata.append({
            "image_id": f"IMG_{image_id:05d}",
            "patient_id": patient_id,
            "modality": random.choice(image_modalities),
            "study_date": study_date,
            "body_site": "Thorax",
            "finding": random.choice(findings),
            "slice_count": random.randint(80, 400) if random.random() > 0.3 else None
        })
        image_id += 1

json_content = json.dumps({"patients": patient_ids, "images": images_metadata}, indent=2)
json_path = f"{OUTPUT_DIR}/lung_cancer_images_metadata.json"
w.files.upload(file_path=json_path, contents=io.BytesIO(json_content.encode("utf-8")), overwrite=True)
print(f"Wrote {len(images_metadata)} image records to {json_path}")

Wrote 151 image records to /Volumes/melissap/melissa_pang/project_volume/lung_cancer_images_metadata.json


In [17]:
# Generate Electronic Health Records CSV (1 row per patient)
import io
import pandas as pd

Faker.seed(SEED + 2)
fake = Faker()

ehr_rows = []
for patient_id in patient_ids:
    dob = fake.date_of_birth(minimum_age=45, maximum_age=85)
    sex = fake.random_element(["M", "F"])
    ehr_rows.append({
        "patient_id": patient_id,
        "date_of_birth": dob.isoformat(),
        "sex": sex,
        "diagnosis": fake.random_element([
            "Non-small cell lung cancer", "Small cell lung cancer", "Lung adenocarcinoma"
        ]),
        "stage": fake.random_element(["I", "II", "III", "IV"]),
        "diagnosis_date": fake.date_between(start_date="-3y", end_date="-1m").isoformat(),
        "medications": fake.sentence(nb_words=6),
        "allergies": fake.sentence(nb_words=3) if fake.boolean(chance_of_getting_true=30) else "NKDA",
        "last_visit": fake.date_between(start_date="-3m", end_date="today").isoformat(),
    })

ehr_df = pd.DataFrame(ehr_rows)
csv_path = f"{OUTPUT_DIR}/ehr.csv"
csv_content = ehr_df.to_csv(index=False)
w.files.upload(file_path=csv_path, contents=io.BytesIO(csv_content.encode("utf-8")), overwrite=True)
print(f"Wrote EHR for {len(ehr_rows)} patients to {csv_path}")
ehr_df.head()

Wrote EHR for 35 patients to /Volumes/melissap/melissa_pang/project_volume/ehr.csv


,patient_id,date_of_birth,sex,diagnosis,stage,diagnosis_date,medications,allergies,last_visit
0,PAT_001,1956-12-03,M,Non-small cell lung cancer,IV,2023-11-06,Different notice beyond.,NKDA,2026-03-04
1,PAT_002,1980-09-28,F,Lung adenocarcinoma,IV,2025-06-20,Group full executive city memory open.,NKDA,2026-03-04
2,PAT_003,1944-08-04,F,Non-small cell lung cancer,III,2024-04-30,Live course candidate section since analysis w...,NKDA,2026-03-04
3,PAT_004,1941-09-22,F,Small cell lung cancer,IV,2025-01-23,Soon character low year say yet pattern wind.,NKDA,2026-03-04
4,PAT_005,1946-08-16,M,Non-small cell lung cancer,III,2026-02-26,Nor identify guess both difference respond.,NKDA,2026-03-04


In [18]:
# Summary: outputs and linkage by patient_id
print("Summary")
print("-" * 50)
print(f"Patient IDs (link key): {patient_ids[0]} .. {patient_ids[-1]}")
print(f"PDF clinical notes: {NUM_PDF_NOTES} files in volume subfolder: {PDF_SUBDIR}")
print(f"Lung cancer images metadata: {OUTPUT_DIR}/lung_cancer_images_metadata.json")
print(f"EHR: {OUTPUT_DIR}/ehr.csv")
print("-" * 50)
print("Use patient_id to join EHR, images metadata, and clinical note PDFs (filename contains patient_id).")

Summary
--------------------------------------------------
Patient IDs (link key): PAT_001 .. PAT_035
PDF clinical notes: 100 files in volume subfolder: /Volumes/melissap/melissa_pang/project_volume/clinical_notes
Lung cancer images metadata: /Volumes/melissap/melissa_pang/project_volume/lung_cancer_images_metadata.json
EHR: /Volumes/melissap/melissa_pang/project_volume/ehr.csv
--------------------------------------------------
Use patient_id to join EHR, images metadata, and clinical note PDFs (filename contains patient_id).
